# Employee Performance & Payroll Analysis
### Integrated Pandas Practical 2
**Scenario:** As a junior HR data analyst, HR has provided an employee master file containing missing values and suspected duplicate records. This notebook cleans the dataset and summarizes salary and performance by department and city.

**Dataset:** `employee_performance_payroll_practical.csv`

## Part A — Initial Inspection

**A1. Load the CSV into df.**

In [1]:
import pandas as pd

df = pd.read_csv("employee_performance_payroll_practical.csv")
df

,Emp_ID,Name,Department,City,Experience,Salary,Rating
0,E101,Aarav,IT,Surat,3.0,45000.0,4.2
1,E102,Diya,HR,Mumbai,5.0,52000.0,4.5
2,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
3,E104,Priya,IT,Ahmedabad,NaN,60000.0,4.7
4,E105,Arjun,Sales,Mumbai,6.0,NaN,4.1
5,E106,Neha,Finance,Surat,4.0,55000.0,4.3
6,E107,Rahul,HR,Ahmedabad,2.0,41000.0,3.9
7,E108,Kavya,IT,Mumbai,7.0,72000.0,4.8
8,E109,Vivek,Finance,Ahmedabad,5.0,61000.0,4.4
9,E110,Meera,Sales,Surat,3.0,42000.0,4.0


**A2. Display the first eight rows.**

In [2]:
df.head(8)

,Emp_ID,Name,Department,City,Experience,Salary,Rating
0,E101,Aarav,IT,Surat,3.0,45000.0,4.2
1,E102,Diya,HR,Mumbai,5.0,52000.0,4.5
2,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
3,E104,Priya,IT,Ahmedabad,NaN,60000.0,4.7
4,E105,Arjun,Sales,Mumbai,6.0,NaN,4.1
5,E106,Neha,Finance,Surat,4.0,55000.0,4.3
6,E107,Rahul,HR,Ahmedabad,2.0,41000.0,3.9
7,E108,Kavya,IT,Mumbai,7.0,72000.0,4.8


**A3. Find the number of rows and columns.**

In [3]:
rows, cols = df.shape
print(f"Rows: {rows}")
print(f"Columns: {cols}")

Rows: 32
Columns: 7


**A4. Display all column names.**

In [4]:
df.columns.tolist()

['Emp_ID', 'Name', 'Department', 'City', 'Experience', 'Salary', 'Rating']

**A5. Inspect data types and non-null counts.**

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Emp_ID      32 non-null     str    
 1   Name        32 non-null     str    
 2   Department  32 non-null     str    
 3   City        32 non-null     str    
 4   Experience  31 non-null     float64
 5   Salary      31 non-null     float64
 6   Rating      32 non-null     float64
dtypes: float64(3), str(4)
memory usage: 1.9 KB


**A6. Observations on possible data-quality issues:**

1. The `Experience` column has a missing value (row for `E104`, Priya) and the `Salary` column has a missing value (row for `E105`, Arjun) — these need to be imputed before analysis.
2. Two employee records appear twice with identical values across all columns (`E103` – Rohan and `E109` – Vivek), suggesting exact duplicate rows that were likely entered twice during data collection and should be removed.

## Part B — Missing Values

**B1. Count missing values in each column.**

In [6]:
df.isnull().sum()

Emp_ID        0
Name          0
Department    0
City          0
Experience    1
Salary        1
Rating        0
dtype: int64

**B2. Display rows containing at least one missing value.**

In [7]:
df[df.isnull().any(axis=1)]

,Emp_ID,Name,Department,City,Experience,Salary,Rating
3,E104,Priya,IT,Ahmedabad,NaN,60000.0,4.7
4,E105,Arjun,Sales,Mumbai,6.0,NaN,4.1


**B3. Find the total number of missing cells.**

In [8]:
total_missing = df.isnull().sum().sum()
print(f"Total missing cells: {total_missing}")

Total missing cells: 2


**B4. Treat the missing Experience value using a sensible numerical summary.**

`Experience` is a numeric, roughly symmetric column (no extreme outliers), so the **mean** experience across all employees is a reasonable, simple summary to fill the single missing value with — it keeps the overall average experience of the workforce unchanged. (The **median** would also be acceptable here since the values are fairly evenly spread; either choice is defensible for a single missing entry.)

In [9]:
mean_experience = df["Experience"].mean()
print(f"Mean Experience (excluding missing): {mean_experience:.2f}")

df["Experience"] = df["Experience"].fillna(mean_experience)
df.loc[df["Name"] == "Priya"]

Mean Experience (excluding missing): 4.13


,Emp_ID,Name,Department,City,Experience,Salary,Rating
3,E104,Priya,IT,Ahmedabad,4.129032,60000.0,4.7


**B5. Fill the missing Salary using the median Salary of the Sales department.**

In [10]:
sales_median_salary = df.loc[df["Department"] == "Sales", "Salary"].median()
print(f"Median Salary in Sales department: {sales_median_salary}")

df["Salary"] = df["Salary"].fillna(sales_median_salary)
df.loc[df["Name"] == "Arjun"]

Median Salary in Sales department: 43500.0


,Emp_ID,Name,Department,City,Experience,Salary,Rating
4,E105,Arjun,Sales,Mumbai,6.0,43500.0,4.1


**B6. Verify that no missing values remain.**

In [11]:
print(df.isnull().sum())
print(f"\nTotal missing cells now: {df.isnull().sum().sum()}")

Emp_ID        0
Name          0
Department    0
City          0
Experience    0
Salary        0
Rating        0
dtype: int64

Total missing cells now: 0


## Part C — Duplicate Handling

**C1. Count exact duplicate rows.**

In [12]:
exact_dupes = df.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_dupes}")

Number of exact duplicate rows: 2


**C2. Display all rows involved in exact duplication.**

In [13]:
df[df.duplicated(keep=False)]

,Emp_ID,Name,Department,City,Experience,Salary,Rating
2,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
8,E109,Vivek,Finance,Ahmedabad,5.0,61000.0,4.4
20,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
21,E109,Vivek,Finance,Ahmedabad,5.0,61000.0,4.4


**C3. Check repeated Emp_ID values using subset.**

In [14]:
df[df.duplicated(subset='Emp_ID', keep=False)].sort_values('Emp_ID')

,Emp_ID,Name,Department,City,Experience,Salary,Rating
2,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
20,E103,Rohan,Sales,Surat,2.0,38000.0,3.8
8,E109,Vivek,Finance,Ahmedabad,5.0,61000.0,4.4
21,E109,Vivek,Finance,Ahmedabad,5.0,61000.0,4.4


**C4. Why repeated Department or City values are not duplicate employee records.**

`Department` and `City` are categorical attributes that many different employees legitimately share (e.g., several employees can work in "IT" or be based in "Surat"). A duplicate *row* only exists when **every** column — including the unique `Emp_ID`, `Name`, `Experience`, `Salary`, and `Rating` — matches exactly across two rows. Repetition in just `Department` or `City` reflects normal group membership, not duplicated data entry.

**C5. Remove confirmed exact duplicates and reset the index.**

In [15]:
df = df.drop_duplicates().reset_index(drop=True)
df

,Emp_ID,Name,Department,City,Experience,Salary,Rating
0,E101,Aarav,IT,Surat,3.000000,45000.0,4.2
1,E102,Diya,HR,Mumbai,5.000000,52000.0,4.5
2,E103,Rohan,Sales,Surat,2.000000,38000.0,3.8
3,E104,Priya,IT,Ahmedabad,4.129032,60000.0,4.7
4,E105,Arjun,Sales,Mumbai,6.000000,43500.0,4.1
5,E106,Neha,Finance,Surat,4.000000,55000.0,4.3
6,E107,Rahul,HR,Ahmedabad,2.000000,41000.0,3.9
7,E108,Kavya,IT,Mumbai,7.000000,72000.0,4.8
8,E109,Vivek,Finance,Ahmedabad,5.000000,61000.0,4.4
9,E110,Meera,Sales,Surat,3.000000,42000.0,4.0


**C6. Verify that no exact duplicates remain.**

In [16]:
print(f"Exact duplicate rows remaining: {df.duplicated().sum()}")
print(f"Duplicate Emp_IDs remaining: {df.duplicated(subset='Emp_ID').sum()}")

Exact duplicate rows remaining: 0
Duplicate Emp_IDs remaining: 0


**C7. Report the cleaned number of employees.**

In [17]:
print(f"Cleaned number of employees: {len(df)}")

Cleaned number of employees: 30


## Part D — Sorting

**D1. Sort employees by Salary from highest to lowest.**

In [18]:
df_sorted_salary = df.sort_values('Salary', ascending=False)
df_sorted_salary

,Emp_ID,Name,Department,City,Experience,Salary,Rating
12,E113,Kabir,Finance,Surat,8.000000,78000.0,4.9
25,E126,Simran,Finance,Ahmedabad,7.000000,73000.0,4.8
7,E108,Kavya,IT,Mumbai,7.000000,72000.0,4.8
26,E127,Harsh,IT,Mumbai,6.000000,69000.0,4.6
17,E118,Riya,Sales,Ahmedabad,7.000000,68000.0,4.7
21,E122,Nisha,Finance,Surat,6.000000,67000.0,4.6
14,E115,Aditya,IT,Mumbai,5.000000,65000.0,4.5
8,E109,Vivek,Finance,Ahmedabad,5.000000,61000.0,4.4
3,E104,Priya,IT,Ahmedabad,4.129032,60000.0,4.7
29,E130,Zoya,Finance,Mumbai,4.000000,59000.0,4.3


**D2. Display the five highest-paid employees.**

In [19]:
df.sort_values('Salary', ascending=False).head(5)

,Emp_ID,Name,Department,City,Experience,Salary,Rating
12,E113,Kabir,Finance,Surat,8.0,78000.0,4.9
25,E126,Simran,Finance,Ahmedabad,7.0,73000.0,4.8
7,E108,Kavya,IT,Mumbai,7.0,72000.0,4.8
26,E127,Harsh,IT,Mumbai,6.0,69000.0,4.6
17,E118,Riya,Sales,Ahmedabad,7.0,68000.0,4.7


**D3. Display the five employees with the highest Rating.**

In [20]:
df.sort_values('Rating', ascending=False).head(5)

,Emp_ID,Name,Department,City,Experience,Salary,Rating
12,E113,Kabir,Finance,Surat,8.000000,78000.0,4.9
7,E108,Kavya,IT,Mumbai,7.000000,72000.0,4.8
25,E126,Simran,Finance,Ahmedabad,7.000000,73000.0,4.8
3,E104,Priya,IT,Ahmedabad,4.129032,60000.0,4.7
17,E118,Riya,Sales,Ahmedabad,7.000000,68000.0,4.7


**D4. Sort first by Department, then by Salary (highest to lowest) within each department.**

In [21]:
df.sort_values(['Department', 'Salary'], ascending=[True, False])

,Emp_ID,Name,Department,City,Experience,Salary,Rating
12,E113,Kabir,Finance,Surat,8.000000,78000.0,4.9
25,E126,Simran,Finance,Ahmedabad,7.000000,73000.0,4.8
21,E122,Nisha,Finance,Surat,6.000000,67000.0,4.6
8,E109,Vivek,Finance,Ahmedabad,5.000000,61000.0,4.4
29,E130,Zoya,Finance,Mumbai,4.000000,59000.0,4.3
5,E106,Neha,Finance,Surat,4.000000,55000.0,4.3
16,E117,Dev,Finance,Mumbai,2.000000,43000.0,3.8
11,E112,Anaya,HR,Mumbai,6.000000,58000.0,4.6
27,E128,Ira,HR,Surat,5.000000,54000.0,4.4
1,E102,Diya,HR,Mumbai,5.000000,52000.0,4.5


**D5. Find the three least-experienced employees.**

In [22]:
df.nsmallest(3, 'Experience')

,Emp_ID,Name,Department,City,Experience,Salary,Rating
10,E111,Ishaan,IT,Surat,1.0,36000.0,3.7
19,E120,Tara,HR,Ahmedabad,1.0,35000.0,3.6
24,E125,Karan,Sales,Surat,1.0,33000.0,3.5


**D6. Find the highest-paid employee.**

In [23]:
df.loc[df['Salary'].idxmax()]

Emp_ID           E113
Name            Kabir
Department    Finance
City            Surat
Experience        8.0
Salary        78000.0
Rating            4.9
Name: 12, dtype: object

## Part E — Grouping

**E1. Count employees in each Department.**

In [24]:
df.groupby('Department')['Emp_ID'].count()

Department
Finance    7
HR         7
IT         8
Sales      8
Name: Emp_ID, dtype: int64

**E2. Calculate average Salary by Department.**

In [25]:
df.groupby('Department')['Salary'].mean().round(2)

Department
Finance    62285.71
HR         47714.29
IT         55500.00
Sales      46562.50
Name: Salary, dtype: float64

**E3. Calculate minimum, maximum, and average Salary by Department.**

In [26]:
df.groupby('Department')['Salary'].agg(['min', 'max', 'mean']).round(2)

,min,max,mean
Department,,,
Finance,43000.0,78000.0,62285.71
HR,35000.0,58000.0,47714.29
IT,36000.0,72000.0,55500.00
Sales,33000.0,68000.0,46562.50


**E4. Calculate average Rating by Department.**

In [27]:
df.groupby('Department')['Rating'].mean().round(2)

Department
Finance    4.44
HR         4.19
IT         4.34
Sales      4.09
Name: Rating, dtype: float64

**E5. Calculate average Salary by City.**

In [28]:
df.groupby('City')['Salary'].mean().round(2)

City
Ahmedabad    52222.22
Mumbai       56750.00
Surat        49909.09
Name: Salary, dtype: float64

**E6. Find maximum Salary in each Department.**

In [29]:
df.groupby('Department')['Salary'].max()

Department
Finance    78000.0
HR         58000.0
IT         72000.0
Sales      68000.0
Name: Salary, dtype: float64

**E7. Group by both City and Department and calculate average Salary.**

In [30]:
df.groupby(['City', 'Department'])['Salary'].mean().round(2)

City       Department
Ahmedabad  Finance       67000.00
           HR            38000.00
           IT            50000.00
           Sales         53333.33
Mumbai     Finance       51000.00
           HR            53333.33
           IT            68666.67
           Sales         49750.00
Surat      Finance       66666.67
           HR            49000.00
           IT            46000.00
           Sales         37666.67
Name: Salary, dtype: float64

**E8. Identify the Department with the highest average Salary.**

In [31]:
dept_avg_salary = df.groupby('Department')['Salary'].mean()
top_salary_dept = dept_avg_salary.idxmax()
print(f"Department with highest average Salary: {top_salary_dept} (₹{dept_avg_salary.max():.2f})")

Department with highest average Salary: Finance (₹62285.71)


**E9. Identify the Department with the highest average Rating.**

In [32]:
dept_avg_rating = df.groupby('Department')['Rating'].mean()
top_rating_dept = dept_avg_rating.idxmax()
print(f"Department with highest average Rating: {top_rating_dept} ({dept_avg_rating.max():.2f})")

Department with highest average Rating: Finance (4.44)


## Part F — Management Questions

**F1. Which department has the largest number of employees?**

In [33]:
dept_counts = df.groupby('Department')['Emp_ID'].count()
print(dept_counts)
print(f"\nLargest headcount: {dept_counts.idxmax()} ({dept_counts.max()} employees)")

Department
Finance    7
HR         7
IT         8
Sales      8
Name: Emp_ID, dtype: int64

Largest headcount: IT (8 employees)


**F2. Which department has the highest average Salary?**

In [34]:
dept_avg_salary = df.groupby('Department')['Salary'].mean().round(2)
print(dept_avg_salary)
print(f"\nHighest average Salary: {dept_avg_salary.idxmax()} (₹{dept_avg_salary.max():.2f})")

Department
Finance    62285.71
HR         47714.29
IT         55500.00
Sales      46562.50
Name: Salary, dtype: float64

Highest average Salary: Finance (₹62285.71)


**F3. Which department has the highest average Rating?**

In [35]:
dept_avg_rating = df.groupby('Department')['Rating'].mean().round(2)
print(dept_avg_rating)
print(f"\nHighest average Rating: {dept_avg_rating.idxmax()} ({dept_avg_rating.max():.2f})")

Department
Finance    4.44
HR         4.19
IT         4.34
Sales      4.09
Name: Rating, dtype: float64

Highest average Rating: Finance (4.44)


**F4. Which city has the highest average Salary?**

In [36]:
city_avg_salary = df.groupby('City')['Salary'].mean().round(2)
print(city_avg_salary)
print(f"\nHighest average Salary: {city_avg_salary.idxmax()} (₹{city_avg_salary.max():.2f})")

City
Ahmedabad    52222.22
Mumbai       56750.00
Surat        49909.09
Name: Salary, dtype: float64

Highest average Salary: Mumbai (₹56750.00)


**F5. Who is the highest-paid employee?**

In [37]:
df.loc[df['Salary'].idxmax(), ['Emp_ID', 'Name', 'Department', 'City', 'Salary']]

Emp_ID           E113
Name            Kabir
Department    Finance
City            Surat
Salary        78000.0
Name: 12, dtype: object

**F6. Who are the top three employees by Rating?**

In [38]:
df.nlargest(3, 'Rating')[['Emp_ID', 'Name', 'Department', 'Rating']]

,Emp_ID,Name,Department,Rating
12,E113,Kabir,Finance,4.9
7,E108,Kavya,IT,4.8
25,E126,Simran,Finance,4.8


**F7. How many employees have a Rating of 4.5 or above?**

In [39]:
high_raters = df[df['Rating'] >= 4.5]
print(f"Employees with Rating >= 4.5: {len(high_raters)}")
high_raters[['Emp_ID', 'Name', 'Department', 'Rating']]

Employees with Rating >= 4.5: 10


,Emp_ID,Name,Department,Rating
1,E102,Diya,HR,4.5
3,E104,Priya,IT,4.7
7,E108,Kavya,IT,4.8
11,E112,Anaya,HR,4.6
12,E113,Kabir,Finance,4.9
14,E115,Aditya,IT,4.5
17,E118,Riya,Sales,4.7
21,E122,Nisha,Finance,4.6
25,E126,Simran,Finance,4.8
26,E127,Harsh,IT,4.6


**F8. Display employees earning more than 60,000, sorted from highest to lowest Salary.**

In [40]:
df[df['Salary'] > 60000].sort_values('Salary', ascending=False)[['Emp_ID', 'Name', 'Department', 'Salary']]

,Emp_ID,Name,Department,Salary
12,E113,Kabir,Finance,78000.0
25,E126,Simran,Finance,73000.0
7,E108,Kavya,IT,72000.0
26,E127,Harsh,IT,69000.0
17,E118,Riya,Sales,68000.0
21,E122,Nisha,Finance,67000.0
14,E115,Aditya,IT,65000.0
8,E109,Vivek,Finance,61000.0


**F9. Which department appears strongest on both salary and rating? Support your answer with numbers.**

In [41]:
dept_summary = df.groupby('Department').agg(
    Avg_Salary=('Salary', 'mean'),
    Avg_Rating=('Rating', 'mean'),
    Headcount=('Emp_ID', 'count')
).round(2).sort_values('Avg_Salary', ascending=False)
dept_summary

,Avg_Salary,Avg_Rating,Headcount
Department,,,
Finance,62285.71,4.44,7
IT,55500.00,4.34,8
HR,47714.29,4.19,7
Sales,46562.50,4.09,8


*Interpretation:* Comparing average Salary and average Rating side by side (see table above), the **Finance** department leads on both measures — it has the highest average Salary as well as the highest average Rating among all departments. This makes Finance the department that appears strongest overall on both dimensions.

**F10. Why should HR inspect duplicate Emp_ID values before deleting records automatically?**

Automatically deleting every row that shares an `Emp_ID` is risky because:
- Two rows can share the same `Emp_ID` due to a genuine **data-entry duplication error** (the same employee record typed in twice) — these should be removed.
- However, a repeated `Emp_ID` could also indicate a **data-correction entry** (e.g., a salary revision or role change logged as a new row) or a **typo in the ID** for what is actually a different employee — blindly deleting these would wrongly erase valid, distinct records.

HR should manually inspect the full rows behind each duplicate `Emp_ID` (not just the ID) to confirm whether the records are truly identical duplicates or legitimate distinct entries before removing anything, to avoid losing real employee data.

## Summary

- Missing values in `Experience` and `Salary` were treated using the mean Experience and the Sales-department median Salary respectively.
- Two exact duplicate rows (`E103`, `E109`) were identified and removed, leaving a clean dataset of unique employees.
- **Finance** stands out as the department with the highest average Salary and the highest average Rating, making it the strongest-performing department on both fronts.
- **Mumbai** and **IT**/other groupings can be compared further using the City × Department pivot in Part E for more granular HR planning.